## Lab Overview: Machine Learning Modeling

Once we understand our data, the next step is to build a machine learning model that can learn patterns from the data and use them to make predictions. In this lab, we will apply supervised machine learning techniques to the DNS data to develop a model for distinguishing **benign** traffic from **attack** traffic.

In this lab, modeling serves several important purposes:

| Function | Description |
|---|---|
| **Prepare the data** | Transform features and labels into a form suitable for machine learning |
| **Train a model** | Use labeled data to learn patterns that distinguish benign from attack traffic |
| **Evaluate performance** | Measure how well the model performs on previously unseen data |
| **Compare models** | Examine how different modeling approaches perform on the classification task |

These activities help us answer important questions about our modeling problem, such as:

- Can we use the available DNS features to distinguish benign from attack traffic?
- How well does a model generalize to data it has not seen during training?
- What types of errors does the model make?
- Which evaluation metrics are most appropriate for this classification problem?
- How does model performance change when we use different modeling approaches or feature sets?

> **Building a model is not the same as building a useful model**. A model can perform well on training data while performing poorly on new observations. We therefore need to carefully separate training and evaluation data and use appropriate performance measures to assess how well the model generalizes.

In the remainder of this lab, we will progressively prepare the DNS data, train machine learning classifiers, and evaluate their performance. We will focus on what the models predict and how well they perform; in the next lab, we will examine why the models make those predictions using explainability techniques.

## DNS Data Exfiltration Challenge Problem Overview

In this lab, your objective is to fit ML models to DNS traffic contained in the **CIC-Bell-DNS-EXF-2021** dataset. Developed in collaboration with Bell Canada Cyber Threat Intelligence (CTI), this dataset focuses specifically on identifying DNS-based data exfiltration and covert command-and-control (C2) tunneling—common techniques used by adversaries to stealthily extract sensitive information from enterprise networks. Because DNS is an essential service, it is typically allowed through firewalls, making it an attractive channel for cybercriminals to encode and exfiltrate data without raising alarms.

For more details on the dataset, visit: [https://www.unb.ca/cic/datasets/dns-exf-2021.html](https://www.unb.ca/cic/datasets/dns-exf-2021.html)

## 1. Setup and Data Loading

In [31]:
import pandas as pd

# Handle Colab environment
colab = True
if colab:
    file_path = "https://raw.githubusercontent.com/mmw188/ISW_2026/main/data/Exfil_Data.csv"
else:
    file_path = './data/Exfil_Data.csv'

# Load data
data = pd.read_csv(file_path)

### Feature Definitions

The data consist of the following features:

| Feature | Data Type | Description |
|---|---|---|
| `FQDN_count` | Integer | Total number of characters in the fully qualified domain name (FQDN). |
| `subdomain_length` | Integer | Number of characters in the subdomain. |
| `upper` | Integer | Number of uppercase characters. |
| `lower` | Integer | Number of lowercase characters. |
| `numeric` | Integer | Number of numeric characters. |
| `entropy` | Float | Shannon entropy of the query name, measuring randomness/diversity. |
| `special` | Integer | Number of special characters (e.g., `-`, `_`, `=`, spaces). |
| `labels` | Integer | Number of labels in the domain name (separated by periods). |
| `labels_max` | Integer | Maximum length of any individual domain label. |
| `labels_average` | Float | Average length of the domain labels. |
| `longest_word` | Float | Ratio of the longest meaningful word to the total domain length. |
| `sld` | String | Second-level domain (SLD). |
| `len` | Integer | Total length of the domain and subdomain. |
| `subdomain` | Boolean | Indicates whether the domain contains a subdomain. |
| `label` | String | Target class/label (`attack` or `benign`). |


### Outcome Definitions

The data contain two classes of outputs.

| Class Label | Description |
|---|---|
| `attack` | DNS queries associated with data exfiltration activity. |
| `benign` | Legitimate DNS queries not associated with data exfiltration activity. |

## 2. Machine Learning Analysis

In this section, we will apply machine learning techniques to detect malicious activity. Using features extracted from the DNS traffic, we will train and evaluate a supervised learning model to classify network behavior as either **benign** or **attack**.

#### 2.1. Data Preparation for Modeling 

Before training a machine learning model, the data must be transformed into a format that the algorithm can use. We also need to separate the data used to learn the model from the data used to evaluate it. This helps us determine whether the model has learned generalizable patterns rather than simply memorizing the training data.

The following sections walk through the data preparation steps.

##### Step 1. Separate features and target.
Split the dataset into the feature variables (X) used for prediction and the target variable (y) that we want the model to predict.

Then, divide the dataset into training and test sets, using 80% of the observations for training and 20% for testing. The training data will be used to learn patterns, while the test data will be reserved for evaluating how well the model generalizes to unseen observations.

In [ ]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = data.drop(columns=["label"])
y = data["label"]

# Split into training and test sets
# Stratification preserves the class distribution in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

##### Step 2. Encode the target labels.
Convert the target labels into the format required by the classifier.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode the target labels
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

##### Step 3. Encode categorical features.
Use a category encoder to convert non-numeric feature columns into a numeric representation suitable for machine learning algorithms.

In [ ]:
import category_encoders as ce

# Identify categorical features
categorical_columns = ["sld"]

# Encode categorical features
# Fit the encoder using the training data only.
encoder = ce.TargetEncoder(
    cols=categorical_columns
)

X_train_encoded = encoder.fit_transform(
    X_train,
    y_train_encoded,
)

# Apply the same learned transformation to the test data.
X_test_encoded = encoder.transform(
    X_test
)

# Ensure all features are numeric
X_train_encoded = X_train_encoded.apply(
    pd.to_numeric,
    errors="coerce",
)

X_test_encoded = X_test_encoded.apply(
    pd.to_numeric,
    errors="coerce",
)

print(f"Training samples:   {len(X_train_encoded):,}")
print(f"Test samples:       {len(X_test_encoded):,}")
print(f"Number of features: {X_train_encoded.shape[1]}")

#### 2.2. Training the Random Forest Model

Training a supervised learning model involves using labeled training data to learn a relationship between the input features and the target variable. Once trained, the model can use those learned patterns to make predictions about observations it has not seen before. The following sections walk through the data modeling steps.

##### Step 1. Instantiate the classifier.
Create a Random Forest classifier using the default model parameters and a fixed random state to make the results reproducible.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the Random Forest classifier
rf_classifier = RandomForestClassifier(random_state=42)

##### Step 2. Train the classifier.
Fit the Random Forest to the training features and target labels so that it can learn patterns associated with benign and attack traffic.

In [ ]:
# Train the classifier
rf_classifier.fit(X_train_encoded, y_train_encoded)

##### Step 3. Generate predictions.
Use the trained classifier to predict labels for the previously unseen test features. The predictions generated here will be evaluated in the next section using several performance metrics and a confusion matrix.

In [ ]:
# Generate predictions on the test set
y_pred_encoded = rf_classifier.predict(X_test_encoded)

# Convert predictions back to the original class labels
y_pred = label_encoder.inverse_transform(y_pred_encoded)

print("Encoded predictions:", y_pred_encoded[:5])
print("Decoded predictions:", y_pred[:5])

#### 2.3. Evaluating Model Performance

Once the model has been trained and has generated predictions for the test data, we need to determine how well it performs. Model evaluation allows us to compare the predicted labels with the known labels in the test set and quantify the model's ability to distinguish between benign and attack traffic.

A model's performance cannot be judged by a single metric. In an attack detection problem, it is particularly important to understand not only how often the model is correct, but also what types of errors it makes.

We will consider several complementary measures:

| Metric | Description |
|---|---|
| **Accuracy** | The overall proportion of observations classified correctly. |
| **Precision** | Of the observations predicted as attacks, the proportion that were actually attacks. |
| **Recall** | Of the actual attacks, the proportion that the model successfully detected. |
| **F1-score** | A combined measure of precision and recall. |
| **Confusion matrix** | A visual summary of correct classifications and the two types of classification errors. |

These measures help us evaluate different aspects of the model. For example, a model may achieve high accuracy while still failing to detect a substantial number of attacks. In a security context, these missed attacks—**false negatives**—may be particularly important.

The confusion matrix provides additional context by showing how many observations were correctly classified and how many were incorrectly classified as the opposite class.

In this section, we will calculate these performance measures and examine the confusion matrix to assess how effectively the Random Forest model detects malicious DNS traffic.

> **The goal of model evaluation is not simply to determine whether the model is correct, but to understand how well it performs and what types of errors it makes.**

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# Calculate accuracy
accuracy = accuracy_score(
    y_test,
    y_pred,
)

print(f"Accuracy: {accuracy:.4f}")

# Generate classification report
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
    )
)

# Calculate and display confusion matrix
cm = confusion_matrix(
    y_test,
    y_pred,
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=label_encoder.classes_,
)

disp.plot()

plt.title("Random Forest Confusion Matrix")
plt.tight_layout()
plt.show()

#### 2.4. Assignment

Now that we have trained and evaluated a Random Forest classifier, explore how other supervised learning algorithms perform on the same classification problem.

1. **Explore other models.**  
   Select one or more supervised learning algorithms from the [scikit-learn documentation](https://scikit-learn.org/stable/supervised_learning.html). Consider models such as:
   - Logistic Regression
   - Decision Tree
   - Support Vector Machine (SVM)
   - K-Nearest Neighbors (KNN)
   - Gradient Boosting

2. **Train the models.**  
   Apply the same training and test data used for the Random Forest model. Train each selected model to classify observations as **benign** or **attack**.

3. **Evaluate the models.**  
   Use the evaluation techniques from Section 2.3 to assess each model. Compare:
   - Accuracy
   - Precision
   - Recall
   - F1-score
   - Confusion matrix

4. **Compare the results.**  
   Consider:
   - Which model performs best overall?
   - Which model is best at detecting attacks?
   - Which models produce the most false positives or false negatives?
   - Are there meaningful differences between the models?

The goal is to understand that **different supervised learning algorithms can learn different decision boundaries from the same data**. Model selection should therefore be guided by both the performance of the models and the requirements of the detection problem.

> **Don't just ask which model has the highest accuracy. Ask which model performs best for the aspects of attack detection that matter most.**